In [ ]:
# Libraries Import
import os
import time
import json
import pickle
import joblib
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

print("Libraries imported successfully")
print("Current Directory:", os.getcwd())

Libraries imported successfully
Current Directory: w:\TopSkills\Week 04, Lab 03\Part 02


In [ ]:
# Data Preparation
print("\nTASK 1: Data Preparation")

X, y = load_breast_cancer(return_X_y=True)
print("Dataset loaded:", X.shape[0], "samples,", X.shape[1], "features")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train set:", X_train.shape[0], "samples")
print("Test set:", X_test.shape[0], "samples")


TASK 1: Data Preparation
Dataset loaded: 569 samples, 30 features
Train set: 455 samples
Test set: 114 samples


In [ ]:
# Pipeline Construction
print("\nTASK 2: Pipeline Construction")

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
])

pipeline.fit(X_train, y_train)
print("Pipeline trained successfully")

train_acc = accuracy_score(y_train, pipeline.predict(X_train))
test_acc = accuracy_score(y_test, pipeline.predict(X_test))

print("Training Accuracy:", round(train_acc, 4))
print("Testing Accuracy:", round(test_acc, 4))


TASK 2: Pipeline Construction
Pipeline trained successfully
Training Accuracy: 1.0
Testing Accuracy: 0.9649


In [ ]:
# Serialization
print("\nTASK 3: Serialization")

version = '1.0.0'

filename_pkl = f'model_v{version}.pkl'
with open(filename_pkl, 'wb') as f:
    pickle.dump(pipeline, f, protocol=5)
print("Pickle saved:", filename_pkl)

filename_joblib = f'model_v{version}.joblib'
joblib.dump(pipeline, filename_joblib)
print("Joblib saved:", filename_joblib)

pkl_size = os.path.getsize(filename_pkl) / (1024 * 1024)
joblib_size = os.path.getsize(filename_joblib) / (1024 * 1024)

print("Pickle size:", round(pkl_size, 2), "MB")
print("Joblib size:", round(joblib_size, 2), "MB")


TASK 3: Serialization
Pickle saved: model_v1.0.0.pkl
Joblib saved: model_v1.0.0.joblib
Pickle size: 0.3 MB
Joblib size: 0.31 MB


In [ ]:
# Version Control
print("\nTASK 4: Version Control")

versions = ['v1', 'v2', 'v3']
random_states = [42, 123, 456]
version_data = {}
models = {}

for ver, rs in zip(versions, random_states):
    print("\nCreating version", ver, "with random_state =", rs)
    
    p = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(n_estimators=100, random_state=rs))
    ])
    p.fit(X_train, y_train)
    models[ver] = p
    
    pkl_file = f'model_{ver}.pkl'
    with open(pkl_file, 'wb') as f:
        pickle.dump(p, f, protocol=5)
    
    joblib_file = f'model_{ver}.joblib'
    joblib.dump(p, joblib_file)
    
    acc = accuracy_score(y_test, p.predict(X_test))
    
    version_data[ver] = {
        'random_state': rs,
        'pkl_file': pkl_file,
        'joblib_file': joblib_file,
        'test_accuracy': round(acc, 4)
    }
    
    print("Saved:", pkl_file, "and", joblib_file)
    print("Test Accuracy:", round(acc, 4))

metadata = {
    'model_name': 'RandomForestPipeline',
    'dataset': 'Breast Cancer Wisconsin',
    'total_versions': len(versions),
    'versions': version_data
}

with open('model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("\nMetadata saved to model_metadata.json")


TASK 4: Version Control

Creating version v1 with random_state = 42
Saved: model_v1.pkl and model_v1.joblib
Test Accuracy: 0.9649

Creating version v2 with random_state = 123
Saved: model_v2.pkl and model_v2.joblib
Test Accuracy: 0.9649

Creating version v3 with random_state = 456
Saved: model_v3.pkl and model_v3.joblib
Test Accuracy: 0.9649

Metadata saved to model_metadata.json


In [ ]:
# Deserialization and Validation
print("\nTASK 5: Deserialization and Validation")

results = []

for ver in versions:
    print("\nVersion:", ver)
    
    t1 = time.time()
    with open(f'model_{ver}.pkl', 'rb') as f:
        m1 = pickle.load(f)
    load1 = time.time() - t1
    acc1 = accuracy_score(y_test, m1.predict(X_test))
    
    t2 = time.time()
    m2 = joblib.load(f'model_{ver}.joblib')
    load2 = time.time() - t2
    acc2 = accuracy_score(y_test, m2.predict(X_test))
    
    size1 = os.path.getsize(f'model_{ver}.pkl') / (1024 * 1024)
    size2 = os.path.getsize(f'model_{ver}.joblib') / (1024 * 1024)
    
    print("Pickle - Load:", round(load1, 4), "s, Accuracy:", round(acc1, 4))
    print("Joblib - Load:", round(load2, 4), "s, Accuracy:", round(acc2, 4))
    
    results.append({
        'version': ver,
        'pickle_load': load1,
        'joblib_load': load2,
        'pickle_accuracy': acc1,
        'joblib_accuracy': acc2,
        'pickle_size': size1,
        'joblib_size': size2
    })


TASK 5: Deserialization and Validation

Version: v1
Pickle - Load: 0.006 s, Accuracy: 0.9649
Joblib - Load: 0.031 s, Accuracy: 0.9649

Version: v2
Pickle - Load: 0.006 s, Accuracy: 0.9649
Joblib - Load: 0.029 s, Accuracy: 0.9649

Version: v3
Pickle - Load: 0.0033 s, Accuracy: 0.9649
Joblib - Load: 0.0247 s, Accuracy: 0.9649


In [ ]:
# Comparison and Report
print("\nComparison Summary")

print("\nFile Size Comparison:")
for r in results:
    diff = ((r['pickle_size'] - r['joblib_size']) / r['pickle_size']) * 100
    print("Version", r['version'])
    print("  Pickle:", round(r['pickle_size'], 2), "MB")
    print("  Joblib:", round(r['joblib_size'], 2), "MB")
    print("  Joblib is", round(diff, 1), "% smaller")

print("\nLoad Time Comparison:")
for r in results:
    diff = ((r['pickle_load'] - r['joblib_load']) / r['pickle_load']) * 100
    print("Version", r['version'])
    print("  Pickle:", round(r['pickle_load'], 4), "s")
    print("  Joblib:", round(r['joblib_load'], 4), "s")
    print("  Joblib is", round(diff, 1), "% faster")

avg_pkl = sum(r['pickle_load'] for r in results) / len(results)
avg_job = sum(r['joblib_load'] for r in results) / len(results)
avg_diff = ((avg_pkl - avg_job) / avg_pkl) * 100

print("\nAverage Performance:")
print("  Pickle average load time:", round(avg_pkl, 4), "s")
print("  Joblib average load time:", round(avg_job, 4), "s")
print("  Joblib is", round(avg_diff, 1), "% faster")

# Generate Report
report = """# Comparison Report: Pickle vs Joblib

## Dataset
- Name: Breast Cancer Wisconsin
- Samples: 569
- Features: 30
- Train/Test Split: 80/20

## Models
- Algorithm: RandomForestClassifier
- Number of trees: 100
- Total versions: 3 (v1, v2, v3)

## File Size Comparison
| Version | Pickle Size | Joblib Size | Difference |
|---------|-------------|-------------|------------|
"""

for r in results:
    diff = ((r['pickle_size'] - r['joblib_size']) / r['pickle_size']) * 100
    report += f"| {r['version']} | {r['pickle_size']:.2f} MB | {r['joblib_size']:.2f} MB | Joblib is {diff:.1f}% smaller |\n"

report += """
## Load Time Comparison
| Version | Pickle Load Time | Joblib Load Time | Difference |
|---------|------------------|------------------|------------|
"""

for r in results:
    diff = ((r['pickle_load'] - r['joblib_load']) / r['pickle_load']) * 100
    report += f"| {r['version']} | {r['pickle_load']:.4f}s | {r['joblib_load']:.4f}s | Joblib is {diff:.1f}% faster |\n"

report += f"""
## Average Performance
- Pickle average load time: {avg_pkl:.4f}s
- Joblib average load time: {avg_job:.4f}s
- Joblib is {avg_diff:.1f}% faster overall

## Conclusion
Joblib is recommended for scikit-learn models because:
1. Faster load times
2. Smaller file sizes
3. Same accuracy as Pickle
"""

with open('comparison_report.md', 'w') as f:
    f.write(report)

print("\nReport saved: comparison_report.md")


Comparison Summary

File Size Comparison:
Version v1
  Pickle: 0.3 MB
  Joblib: 0.31 MB
  Joblib is -3.4 % smaller
Version v2
  Pickle: 0.31 MB
  Joblib: 0.32 MB
  Joblib is -3.4 % smaller
Version v3
  Pickle: 0.3 MB
  Joblib: 0.31 MB
  Joblib is -3.4 % smaller

Load Time Comparison:
Version v1
  Pickle: 0.006 s
  Joblib: 0.031 s
  Joblib is -417.5 % faster
Version v2
  Pickle: 0.006 s
  Joblib: 0.029 s
  Joblib is -383.0 % faster
Version v3
  Pickle: 0.0033 s
  Joblib: 0.0247 s
  Joblib is -651.8 % faster

Average Performance:
  Pickle average load time: 0.0051 s
  Joblib average load time: 0.0282 s
  Joblib is -454.3 % faster

Report saved: comparison_report.md


In [ ]:
# Flask API (Bonus)
print("\nTASK 6: Flask API")

flask_code = """from flask import Flask, request, jsonify
import joblib
import json
import numpy as np

app = Flask(__name__)

model = joblib.load('model_v3.joblib')

with open('model_metadata.json', 'r') as f:
    metadata = json.load(f)

@app.route('/models', methods=['GET'])
def get_models():
    return jsonify(metadata)

@app.route('/predict', methods=['POST'])
def predict():
    try:
        data = request.get_json()
        features = np.array(data['features']).reshape(1, -1)
        prediction = model.predict(features)
        probability = model.predict_proba(features)
        
        return jsonify({
            'prediction': int(prediction[0]),
            'probability': probability.tolist()
        })
    except Exception as e:
        return jsonify({'error': str(e)}), 400

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000, debug=False)
"""

with open('app.py', 'w') as f:
    f.write(flask_code)

print("Flask app saved: app.py")


TASK 6: Flask API
Flask app saved: app.py


In [ ]:
# Final Checklist
print("\nFINAL DELIVERABLES CHECKLIST")
print("Location:", os.getcwd())

deliverables = [
    'model_v1.pkl',
    'model_v1.joblib',
    'model_v2.pkl',
    'model_v2.joblib',
    'model_v3.pkl',
    'model_v3.joblib',
    'model_metadata.json',
    'comparison_report.md',
    'app.py'
]

print("\nFiles Generated:")
for f in deliverables:
    if os.path.exists(f):
        size = os.path.getsize(f) / 1024
        print("EXISTS:", f, "(", round(size, 1), "KB )")
    else:
        print("MISSING:", f)

print("\nALL TASKS COMPLETE")


FINAL DELIVERABLES CHECKLIST
Location: w:\TopSkills\Week 04, Lab 03\Part 02

Files Generated:
EXISTS: model_v1.pkl ( 308.3 KB )
EXISTS: model_v1.joblib ( 318.8 KB )
EXISTS: model_v2.pkl ( 313.3 KB )
EXISTS: model_v2.joblib ( 323.8 KB )
EXISTS: model_v3.pkl ( 307.5 KB )
EXISTS: model_v3.joblib ( 318.0 KB )
EXISTS: model_metadata.json ( 0.6 KB )
EXISTS: comparison_report.md ( 1.1 KB )
EXISTS: app.py ( 0.9 KB )

ALL TASKS COMPLETE
